# MUTANG++ Geodesic Sanity Check

For each of 500 sampled peptides:
1. Encode the parent → z, compute decoder-Jacobian SVD.
2. Enumerate MUTANG candidates and score the full Cartesian product
   with `ProjectedDirectionPairwiseSimilarityPotential` (MUTANG++).
3. Take the top-20 % by potential.
4. Build frozen Christoffel symbols Γ at z (regularised pullback metric).
5. For each top mutant, project the aggregate ambient one-hot direction
   `Σ_p e_{p, new_aa}` through `J_H⁺`, integrate a geodesic for unit time,
   decode the endpoint, and compare against the predicted mutant.

Headline question: *what fraction of top-20 % MUTANG++ mutants are reachable
by geodesic traversal in the projected direction?*

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('.').resolve()))
from _mutang_geodesic_helpers import run_one_parent, summarize_results
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import HydrAMPEncoderDecoder

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

encoder_decoder = HydrAMPEncoderDecoder(
    jacobian_mode='approx',
    device=DEVICE,
    jacobian_eps=0.05,
    field_eps=0.05,
)
encoder_decoder.eval()

RESULTS_DIR = Path('results/mutang_geodesic_sanity')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PILOT_N = 25
FULL_N = 500
TOP_FRAC = 0.20
MAX_MUTANTS_PER_PARENT = 50
N_RK4_STEPS = 32
GEODESIC_T1 = 1.0
print('Setup complete.')

## Load the 500 sampled peptides

In [ ]:
PEPTIDES_FILE = Path('../../basic_eps_greedy_rl/inputs/sampled_500_peptides.txt')
all_peptides = [
    line.strip()
    for line in PEPTIDES_FILE.read_text().splitlines()
    if line.strip()
]
MAX_LEN = 25
valid_peptides = [p for p in all_peptides if 1 <= len(p) <= MAX_LEN]
print(f'loaded {len(all_peptides)} lines, {len(valid_peptides)} ≤ {MAX_LEN} aa')
print('length distribution:')
lens = pd.Series([len(p) for p in valid_peptides])
print(lens.describe().to_string())

## Per-parent driver
Wraps `run_one_parent` from the helpers module (MUTANG → MUTANG++ → top-20 %
→ frozen Γ → geodesic for each top mutant) and adds a tqdm progress loop.

In [ ]:
def run_batch(peptides, label, *, save_path=None, rng_seed=SEED):
    rng = random.Random(rng_seed)
    rows = []
    start = time.time()
    pbar = tqdm(peptides, desc=label)
    for p in pbar:
        try:
            parent_rows = run_one_parent(
                encoder_decoder, p,
                top_frac=TOP_FRAC,
                max_mutants=MAX_MUTANTS_PER_PARENT,
                n_steps=N_RK4_STEPS,
                t1=GEODESIC_T1,
                rng=rng,
            )
        except Exception as e:
            print(f'  ! {p[:25]}: {type(e).__name__}: {e}')
            continue
        rows.extend(parent_rows)
        elapsed = time.time() - start
        pbar.set_postfix(rows=len(rows), elapsed=f'{elapsed:.0f}s')
    total = time.time() - start
    print(f'\n{label}: {len(rows)} rows from {len(peptides)} parents in {total/60:.1f} min')
    if save_path is not None:
        df = pd.DataFrame(rows)
        df.to_csv(save_path, index=False)
        print(f'  saved → {save_path}')
    return rows

## Pilot: 25 peptides
Quick first run to confirm the pipeline works and to estimate runtime.
We expect ~10 minutes on CPU.

In [ ]:
pilot_peptides = valid_peptides[:PILOT_N]
pilot_rows = run_batch(
    pilot_peptides, 'pilot-25',
    save_path=RESULTS_DIR / 'pilot_25_results.csv',
)
pilot_summary, pilot_df = summarize_results(pilot_rows)
print(json.dumps(pilot_summary, indent=2))

In [ ]:
# Spot-check: parent #0, top-1 mutant — side-by-side parent / mutant / decoded
if len(pilot_df) > 0:
    spot = pilot_df.sort_values(['parent', 'rank']).groupby('parent').first().head(5)
    for parent, row in spot.iterrows():
        print(f'parent : {parent}')
        print(f'mutant : {row["mutant"]}')
        print(f'decoded: {row["decoded"]}')
        print(f'  n_mutated={row["n_mutated"]}  pos_match={row["pos_match_count"]}/{row["n_mutated"]}  full={row["full_match"]}  hamming={row["hamming"]}  |v|={row["direction_norm"]:.1f}')
        print()

### Pilot interpretation
Inspect the pilot summary above. The key numbers:

- **full_match_rate**: fraction of top-20 % MUTANG++ mutants whose geodesic endpoint *exactly* decodes to the predicted sequence.
- **pos_match_mean**: average fraction of mutated positions correctly recovered at the endpoint (more forgiving).
- **mean_hamming**: edit distance between decoded endpoint and predicted mutant.
- **mean_n_mutated**: typical Hamming distance between predicted mutants and parents.
- **geo_fallback_rate**: fraction of geodesics that diverged numerically (NaN/Inf) and fell back to Euclidean lines.

If `full_match_rate` is ≈ 0 % and `pos_match_mean` is at chance level (~5 % per position),
that's a meaningful negative result: MUTANG++'s projected directions, while useful for *ranking*
mutations by geometric synergy, do **not** correspond to manifold geodesics that literally
reach the predicted mutant peptide. This is consistent with the linearisation argument:
`J_H⁺ · e_mut` amplifies a unit ambient probability shift by the spectral norm of the
pseudo-inverse, producing latent directions that overshoot far past the mutant's basin.

## Full run: 500 peptides
Runs the same pipeline on the full input. Expect ~3 hours on CPU. Results are
checkpointed to `full_500_results.csv` so they survive notebook restarts.

In [ ]:
full_rows = run_batch(
    valid_peptides[:FULL_N], 'full-500',
    save_path=RESULTS_DIR / 'full_500_results.csv',
)
summary, df = summarize_results(full_rows)
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## Aggregate summary

In [ ]:
df = pd.read_csv(RESULTS_DIR / 'full_500_results.csv')
n_rows = len(df)
n_parents = df['parent'].nunique()
print(f'{n_rows} (parent, mutant) rows from {n_parents} unique parents')
print()
print('Overall:')
print(f'  full_match_rate     : {df["full_match"].mean():.4f}')
print(f'  pos_match_mean      : {df["pos_match_frac"].mean():.4f}')
print(f'  mean_hamming        : {df["hamming"].mean():.2f}')
print(f'  mean_n_mutated      : {df["n_mutated"].mean():.2f}')
print(f'  geo_fallback_rate   : {df["geo_fallback"].mean():.4f}')
print(f'  mean_direction_norm : {df["direction_norm"].mean():.2f}')
print()
print('By n_mutated:')
by_n = df.groupby('n_mutated').agg(
    n=('full_match', 'size'),
    full=('full_match', 'mean'),
    posfrac=('pos_match_frac', 'mean'),
    hamming=('hamming', 'mean'),
)
print(by_n.to_string())

In [ ]:
# Stratify by MUTANG++ rank decile within each parent's top-20%
df_ranked = df.copy()
def to_decile(s):
    if s.nunique() < 2:
        return pd.Series(0, index=s.index)
    try:
        return pd.qcut(s, q=min(10, s.nunique()), labels=False, duplicates='drop')
    except Exception:
        return pd.Series(0, index=s.index)
df_ranked['rank_decile'] = df_ranked.groupby('parent')['rank'].transform(to_decile)
by_dec = df_ranked.groupby('rank_decile').agg(
    n=('full_match', 'size'),
    full=('full_match', 'mean'),
    posfrac=('pos_match_frac', 'mean'),
    log_pot=('log_potential', 'mean'),
)
print('By MUTANG++ rank decile within each parent (0 = highest potential):')
print(by_dec.to_string())

In [ ]:
# Histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['pos_match_frac'].dropna(), bins=20, edgecolor='black')
axes[0].set_xlabel('fraction of mutated positions recovered')
axes[0].set_ylabel('count (mutants)')
axes[0].set_title('Per-position match distribution')
axes[1].hist(df['hamming'].clip(upper=25), bins=25, edgecolor='black')
axes[1].set_xlabel('Hamming(decoded, predicted mutant)')
axes[1].set_title('Hamming distance distribution')
axes[2].hist(np.log10(df['direction_norm'].clip(lower=1e-3)), bins=30, edgecolor='black')
axes[2].set_xlabel('log10 ||v|| (projected direction norm)')
axes[2].set_title('Direction-norm distribution')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'distributions.png', dpi=120)
plt.show()

In [ ]:
# Persist final summary
summary, _ = summarize_results(df.to_dict('records'))
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## Headline
*X %* of top-20 % MUTANG++ mutants are reachable by geodesic traversal in the
projected direction (substitute X from `summary['full_match_rate'] * 100`).

Per-position recovery: *Y %* of mutated positions are correctly recovered on
average (substitute Y from `summary['pos_match_mean'] * 100`).